In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install catboost optuna shap -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 26.6 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
from sklearn import set_config
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer # 추가
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [4]:

# [핵심] Scikit-learn 출력을 Pandas DataFrame으로 유지
set_config(transform_output="pandas")

# Configuration Paths
TRAIN_PATH = '/content/drive/MyDrive/스트레스 지수 예측 해커톤 6차/train.csv'
TEST_PATH = '/content/drive/MyDrive/스트레스 지수 예측 해커톤 6차/test.csv'
SUBMISSION_PATH = '/content/drive/MyDrive/스트레스 지수 예측 해커톤 6차/sample_submission.csv'

def main():
    # 1. Load Data
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)

    target_col = 'stress_score'
    id_col = 'ID'

    # 2. Split features and target
    X_train = train.drop(columns=[id_col, target_col])
    y_train = train[target_col]
    X_test = test.drop(columns=[id_col])

    # 3. Define Categorical and Numerical columns
    cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
    num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

    # 4. Build Preprocessing Pipelines
    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median'))
    ])

    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    preprocessor = ColumnTransformer([
        ('num', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols)
    ])

    # 5. Define Stacking Model
    estimators = [
        ('rf', RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)),
        ('xgb', XGBRegressor(n_estimators=500, random_state=42)),
        ('lgbm', LGBMRegressor(n_estimators=500, random_state=42, verbose=-1)),
        ('cat', CatBoostRegressor(n_estimators=500, random_state=42, verbose=0))
    ]

    # [해결 핵심] FunctionTransformer를 사용하여 Ridge에 전달될 때 이름표(Feature Names) 제거
    stacking_model = StackingRegressor(
        estimators=estimators,
        final_estimator=Pipeline([
            ('to_numpy', FunctionTransformer(lambda x: np.asarray(x))),
            ('ridge', Ridge())
        ]),
        cv=5,
        n_jobs=-1
    )

    # 6. Create Final Pipeline
    model_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', stacking_model)
    ])

    # 7. Validation (MAE)
    print('Starting Cross-Validation (MAE)...')
    # cv_scores가 음수로 나오므로 -1을 곱해 실제 MAE 계산
    cv_scores = cross_val_score(model_pipeline, X_train, y_train, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
    print(f'Average MAE: {-cv_scores.mean():.4f}')

    # 8. Train & Predict
    print('Training final model...')
    model_pipeline.fit(X_train, y_train)

    print('Predicting...')
    predictions = model_pipeline.predict(X_test).round(2)

    # 9. Save Submission
    submission = pd.read_csv(SUBMISSION_PATH)
    submission[target_col] = predictions
    submission.to_csv(SUBMISSION_PATH, index=False)
    print(f'Submission saved successfully to: {SUBMISSION_PATH}')

if __name__ == '__main__':
    main()

Starting Cross-Validation (MAE)...
Average MAE: 0.1835
Training final model...


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_names_out` should be defined.
  warnings.warn(warn_msg.format("pandas"))


Predicting...


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_function_transformer.py:311: UserWarning: When `set_output` is configured to be 'pandas', `func` should return a pandas DataFrame to follow the `set_output` API  or `feature_names_out` should be defined.
  warnings.warn(warn_msg.format("pandas"))


Submission saved successfully to: /content/drive/MyDrive/스트레스 지수 예측 해커톤 6차/sample_submission.csv
